In [2]:
from st_router.paths import get_rootdir
from st_router.scoring import label_rows, score, run_mcnemar, build_prefix
from st_router.prompt import make_prompt, run_llm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from dotenv import load_dotenv
import groq
from groq import Groq
from time import sleep
import re
import logging
logging.basicConfig(level=logging.INFO)


ROOT_DIR = get_rootdir()
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROC = ROOT_DIR / "data" / "processed"
DATA_INTERIM = ROOT_DIR / "data" / "interim"

load_dotenv()

True

# Part 3: Partial Prompt

In [3]:
bash = pd.read_parquet(DATA_PROC / "bash_windows.parquet")

df = bash.copy()
df = df[df["window_len"] != 0]

# randomly select one row per user - 267 users
df = df.groupby("unique_key").sample(random_state=42)
df.head()

# Check if idx aligned with empty prompt df
old = pd.read_json(DATA_INTERIM / "empty_prompt_gpt20b.jsonl", lines=True)
assert(set(df.index) == set(old["idx"]))

df["actual"] = df["cmd"]
df = label_rows(df)
df["label"].value_counts()

df = df[df["label"] != "non_command"].copy()

df[["prefix", "rest"]] = build_prefix(df["target"])
(df["rest"] == "").sum()
df = df[df["rest"] != ""].copy()

In [4]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
prompt = make_prompt()

out20 = DATA_INTERIM / "partial_prompt_gpt20b.jsonl"
run_llm(df, prompt, out20, input_col="prefix")

out120 = DATA_INTERIM / "partial_prompt_gptoss120b.jsonl"
run_llm(df, prompt, out120, input_col="prefix", model="openai/gpt-oss-120b")

out_gem = DATA_INTERIM / "partial_prompt_gemini35flash_lite_low.jsonl"
run_llm(df, prompt, out_gem, input_col="prefix", model="gemini-3.5-flash-lite", provider="gemini", reasoning_effort=None, sleep_s=5)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


row 171: rate limited, waiting 15s


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


row 171: rate limited, waiting 30s


KeyboardInterrupt: 

# Evaluation

In [ ]:
from itertools import combinations
import numpy as np

FILES = {
    "gptoss20b":  DATA_INTERIM / "partial_prompt_gptoss20b.jsonl",
    "gptoss120b": DATA_INTERIM / "partial_prompt_gptoss120b.jsonl",
    "gemini35fl": DATA_INTERIM / "partial_prompt_gemini35flash_lite_low.jsonl",  # use your real filename
}

def load_scored(path):
    df = pd.read_json(path, lines=True)
    df = label_rows(df)
    df = df[df["label"] != "non_command"].copy()
    assert df["idx"].is_unique, f"duplicate idx in {path.name}"
    df = df.set_index("idx").sort_index()
    exact, cmd = score(df["prediction"], df["target"])
    df["exact"] = np.asarray(exact, dtype=bool)
    df["cmd"] = np.asarray(cmd, dtype=bool)
    return df[["target", "exact", "cmd"]]

results = {name: load_scored(path) for name, path in FILES.items()}

# Same rows and same targets in every file
ref = next(iter(results.values()))
for name, r in results.items():
    assert r.index.equals(ref.index), f"{name}: different rows"
    assert r["target"].equals(ref["target"]), f"{name}: different targets"

# Baseline: repeat last command, on the same rows
prev_1 = bash.loc[ref.index, "prev_1"].values
b_exact, b_cmd = score(pd.Series(prev_1, index=ref.index), ref["target"])
results["baseline"] = pd.DataFrame(
    {"target": ref["target"],
     "exact": np.asarray(b_exact, dtype=bool),
     "cmd": np.asarray(b_cmd, dtype=bool)},
    index=ref.index,
)

print(f"n = {len(ref)}")
for name, r in results.items():
    print(f"{name:11s} exact {r['exact'].mean():6.1%}   command {r['cmd'].mean():6.1%}")

for a, b in combinations(results, 2):
    print(f"\n{a} vs {b}")
    run_mcnemar(results[a]["exact"], results[b]["exact"], "exact")
    run_mcnemar(results[a]["cmd"], results[b]["cmd"], "command")

n = 198
gptoss20b   exact  19.2%   command  98.5%
gptoss120b  exact  18.9%   command  95.7%
baseline    exact   4.0%   command  40.4%

gptoss20b vs gptoss120b
exact: both right=26, only model=10, only baseline=9, both wrong=140, p=1.000
command: both right=177, only model=5, only baseline=0, both wrong=3, p=0.062

gptoss20b vs baseline
exact: both right=5, only model=33, only baseline=3, both wrong=157, p=0.000
command: both right=79, only model=116, only baseline=1, both wrong=2, p=0.000

gptoss120b vs baseline
exact: both right=4, only model=31, only baseline=3, both wrong=147, p=0.000
command: both right=72, only model=105, only baseline=3, both wrong=5, p=0.000
